In [6]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [5]:
# data preparation
data = load_diabetes()
# take all features
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')
df = pd.concat([X, y], axis=1)
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [7]:
# lets split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# shapes
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((353, 10), (89, 10), (353,), (89,))

In [8]:
# base model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mse

2900.1936284934814

In [10]:
# Hyper parameter tuning
from sklearn.model_selection import GridSearchCV

param_grid = {
    'fit_intercept': [True, False],
    'copy_X': [True, False],
    'positive': [True, False]
}

grid_search = GridSearchCV(LinearRegression(), param_grid, cv=5)
grid_search.fit(X_train, y_train)
grid_search.best_params_

/Users/rahultiwari/Documents/02_Freelancing/rbyte_ai_engineering/ai-env/lib/python3.13/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: divide by zero encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/rahultiwari/Documents/02_Freelancing/rbyte_ai_engineering/ai-env/lib/python3.13/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: overflow encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/rahultiwari/Documents/02_Freelancing/rbyte_ai_engineering/ai-env/lib/python3.13/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: invalid value encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/rahultiwari/Documents/02_Freelancing/rbyte_ai_engineering/ai-env/lib/python3.13/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: divide by zero encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/rahultiwari/Documents/02_Freelancing/rbyte_ai_engineering/ai-env/lib/python3.13/site-packages/scipy/optimize/_nn

{'copy_X': True, 'fit_intercept': True, 'positive': False}

In [11]:
# lets train with best params
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
mse_best = mean_squared_error(y_test, y_pred_best)
mse_best

2900.1936284934814

In [12]:
# lets do the same with knn regressor

# based model
knn_model = KNeighborsRegressor()
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)
mse_knn = mean_squared_error(y_test, y_pred_knn)
mse_knn

3019.075505617978

In [19]:
# tunning knn regressor
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 13],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

grid_search_knn = GridSearchCV(KNeighborsRegressor(), param_grid_knn, cv=5)
grid_search_knn.fit(X_train, y_train)
grid_search_knn.best_params_



{'algorithm': 'auto', 'n_neighbors': 13, 'weights': 'distance'}

In [20]:
# train with best knn params
best_knn_model = grid_search_knn.best_estimator_
y_pred_best_knn = best_knn_model.predict(X_test)
mse_best_knn = mean_squared_error(y_test, y_pred_best_knn)
mse_best_knn

2966.5153014630882

In [21]:
# Lets try with Random search CV
from sklearn.model_selection import RandomizedSearchCV
param_dist_knn = {
    'n_neighbors': np.arange(1, 31),
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

random_search_knn = RandomizedSearchCV(KNeighborsRegressor(), param_distributions=param_dist_knn, n_iter=100, cv=5, random_state=42)
random_search_knn.fit(X_train, y_train)
random_search_knn.best_params_


{'weights': 'distance', 'n_neighbors': np.int64(18), 'algorithm': 'ball_tree'}

In [22]:
# fit the best model
best_random_knn_model = random_search_knn.best_estimator_
y_pred_best_random_knn = best_random_knn_model.predict(X_test)
mse_best_random_knn = mean_squared_error(y_test, y_pred_best_random_knn)
mse_best_random_knn

3018.497547915766

In [23]:
# bayesian optimization for hyperparameter tuning
from skopt import BayesSearchCV

param_space_knn = {
    'n_neighbors': (1, 30),
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}   

bayes_search_knn = BayesSearchCV(KNeighborsRegressor(), search_spaces=param_space_knn, n_iter=100, cv=5, random_state=42)
bayes_search_knn.fit(X_train, y_train)
bayes_search_knn.best_params_


/Users/rahultiwari/Documents/02_Freelancing/rbyte_ai_engineering/ai-env/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.str_('ball_tree'), np.int64(22), np.str_('distance')] before, using random point ['ball_tree', np.int64(12), 'uniform']
  warnings.warn(
/Users/rahultiwari/Documents/02_Freelancing/rbyte_ai_engineering/ai-env/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.str_('ball_tree'), np.int64(21), np.str_('distance')] before, using random point ['kd_tree', np.int64(5), 'distance']
  warnings.warn(
/Users/rahultiwari/Documents/02_Freelancing/rbyte_ai_engineering/ai-env/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.str_('ball_tree'), np.int64(20), np.str_('distance')] before, using random point ['kd_tree', np.int64(25), 'distance']
  warnings.warn(
/Users/rah

OrderedDict([('algorithm', 'brute'),
             ('n_neighbors', 18),
             ('weights', 'distance')])

In [24]:
# train with best knn params
best_knn_model = grid_search_knn.best_estimator_
y_pred_best_knn = best_knn_model.predict(X_test)
mse_best_knn = mean_squared_error(y_test, y_pred_best_knn)
mse_best_knn

2966.5153014630882

### Reference 
- https://medium.com/@aditib259/a-comprehensive-guide-to-hyperparameter-tuning-in-machine-learning-dd9bb8072d02